# 🎓 Machine Learning Project: Personalized AI Tutor via LLM & Reinforcement Learning
## 📌 Step 02: Exploratory Data Analysis (EDA) & Data Understanding

---

### 📖 Objective:
In this step, we analyze student behavioral distributions, accuracy rates across different knowledge components (skills), question interaction density, and response latencies.


### Cell 1: Import Libraries & Configure Visual Themes


In [ ]:
# Cell 1: Import Core Visualization & Data Libraries
%matplotlib inline
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    pass

print("✅ Libraries Imported Successfully")


### Cell 2: Load Dataset with Fallback


In [ ]:
# Cell 2: Dataset Loading
cur_dir = os.path.abspath(".")
DATA_DIR = os.path.join(cur_dir, "data") if os.path.exists("data") else cur_dir
RAW_DATA_PATH = os.path.join(DATA_DIR, "skill_builder_data_corrected.csv")

if not os.path.exists(RAW_DATA_PATH):
    print("Dataset not found locally, downloading benchmark...")
    url = "https://raw.githubusercontent.com/GaoSida/DKT/master/Assistments/skill_builder_data_corrected.csv"
    import urllib.request
    urllib.request.urlretrieve(url, RAW_DATA_PATH)

df = pd.read_csv(RAW_DATA_PATH, low_memory=False, encoding='latin1')
print(f"Dataset Loaded Successfully: {df.shape[0]:,} rows, {df.shape[1]} columns")
print("✅ Dataset Loaded Successfully")


### Cell 3: Student Cohort & Psychometric KPIs


In [ ]:
# Cell 3: Key Performance Indicators (KPIs)
user_col = "user_id" if "user_id" in df.columns else df.columns[1]
skill_col = "skill_name" if "skill_name" in df.columns else "skill_id"
correct_col = "correct" if "correct" in df.columns else "correctness"

unique_students = df[user_col].nunique()
unique_skills = df[skill_col].dropna().nunique()
overall_accuracy = df[correct_col].mean()

print("=" * 55)
print("📊 STUDENT COHORT METRICS:")
print("=" * 55)
print(f"• Total Interactions:        {len(df):,}")
print(f"• Unique Students:           {unique_students:,}")
print(f"• Unique Knowledge Skills:   {unique_skills}")
print(f"• Mean Population Accuracy:  {overall_accuracy * 100:.2f}%")
print(f"• Mean Questions per Student:{len(df) / max(1, unique_students):.1f}")
print("=" * 55)


### Cell 4: Skill-Wise Accuracy & Activity Breakdown


In [ ]:
# Cell 4: Top Skills Breakdown Table
skill_stats = df.groupby(skill_col).agg(
    total_attempts=(correct_col, "count"),
    correctness_rate=(correct_col, "mean")
).sort_values(by="total_attempts", ascending=False)

skill_stats["accuracy_pct"] = (skill_stats["correctness_rate"] * 100).round(2)
print("--- Top 10 Most Active Skills: ---")
display(skill_stats.head(10))


### Cell 5: Visual EDA Charts (Accuracy, Volume & Class Balance)


In [ ]:
# Cell 5: Visual Charts
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Subplot 1: Skill Correctness Rates
top_skills = df.groupby(skill_col)[correct_col].mean().sort_values().tail(8)
axes[0].barh(top_skills.index, top_skills.values * 100, color="#4f46e5", alpha=0.85)
axes[0].set_title("Correctness Rate by Skill (%)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Accuracy (%)")
axes[0].grid(axis="x", linestyle="--", alpha=0.5)

# Subplot 2: Interaction Volume per Skill
skill_counts = df[skill_col].value_counts().head(8)
axes[1].bar(range(len(skill_counts)), skill_counts.values, color="#06b6d4", alpha=0.85)
axes[1].set_xticks(range(len(skill_counts)))
axes[1].set_xticklabels([f"S{i+1}" for i in range(len(skill_counts))])
axes[1].set_title("Student Interaction Volume per Skill", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Interaction Count")
axes[1].grid(axis="y", linestyle="--", alpha=0.5)

# Subplot 3: Target Class Balance
class_counts = df[correct_col].value_counts()
axes[2].pie(class_counts, labels=["Incorrect (0)", "Correct (1)"], autopct="%1.1f%%", colors=["#f43f5e", "#10b981"], startangle=140, explode=(0.04, 0))
axes[2].set_title("Overall Response Correctness Ratio", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()
print("✅ Step 02 EDA Visualizations Completed Successfully!")
